<div style="display: flex; align-items: center;">

  <!-- Logos -->
  <div style="white-space: nowrap;">
    <img 
      src="https://www.upc.edu/comunicacio/ca/identitat/descarrega-arxius-grafics/fitxers-marca-principal/upc-positiu-p3005.png" 
      width="300"
      style="vertical-align: middle;"
    >
    <img 
      src="https://www.hipotecalowcost.com/wp-content/uploads/2019/08/Logo-CaixaBank.png" 
      width="200"
      style="vertical-align: middle;"
    >
  </div>

  <!-- Texto -->
  <div style="margin-left: auto; margin-right: 10px; text-align: right;">
      <p style="margin: 0;"><b>CaixaBank · Advanced Analytics Program</b></p>
      <p style="margin: 0;"><b>Model Risk & Data Science Training</b></p>
      <p style="margin: 0;">Python profesional para analítica avanzada</p>
  </div>

</div>

# **Threading, multiprocessing, Numba y paralelización**

## Elegir la estrategia correcta según el cuello de botella

En este laboratorio estudiaremos varias formas de acelerar tareas en Python.

La idea principal será:

> **no existe una única técnica de paralelización válida para todos los problemas.**

Antes de elegir una herramienta debemos saber si la tarea es:

- I/O-bound;
- CPU-bound;
- vectorizable;
- compilable con JIT;
- paralelizable;
- suficientemente grande como para compensar el overhead.

Seguiremos la metodología:

> **provocar lentitud → clasificar el problema → medir baseline → aplicar estrategia → volver a medir → validar resultados**

## Objetivos

Al finalizar deberías poder:

1. Diferenciar I/O-bound y CPU-bound.
2. Entender el papel del GIL a nivel conceptual.
3. Utilizar `ThreadPoolExecutor`.
4. Utilizar `ProcessPoolExecutor`.
5. Entender el overhead de procesos.
6. Detectar cuándo multiprocessing empeora el rendimiento.
7. Diseñar benchmarks justos.
8. Comprender compilación JIT.
9. Utilizar `numba.njit`.
10. Entender el coste de warm-up.
11. Comparar Python, NumPy y Numba.
12. Utilizar `parallel=True` y `prange`.
13. Detectar operaciones incompatibles con Numba.
14. Evitar paralelización prematura.
15. Validar equivalencia entre implementaciones.
16. Construir una matriz de decisión para escoger técnica.

## 1. Preparación

In [ ]:
import os
import time
from concurrent.futures import (
    ThreadPoolExecutor,
    ProcessPoolExecutor
)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
rng = np.random.default_rng(SEED)

print("CPUs disponibles:", os.cpu_count())

try:
    from numba import njit, prange
    NUMBA_DISPONIBLE = True
    print("Numba disponible")
except ImportError:
    NUMBA_DISPONIBLE = False
    print("Numba no instalado. Ejecuta: pip install numba")

# 2. Función de benchmark

In [ ]:
def benchmark(
    funcion,
    repeticiones=5,
    calentamiento=1
):
    for _ in range(calentamiento):
        funcion()

    tiempos = []

    for _ in range(repeticiones):
        inicio = time.perf_counter()
        funcion()
        tiempos.append(
            time.perf_counter() - inicio
        )

    tiempos = np.array(tiempos)

    return {
        "media_ms": tiempos.mean() * 1000,
        "mediana_ms": np.median(tiempos) * 1000,
        "std_ms": tiempos.std() * 1000,
        "raw": tiempos
    }

# 3. I/O-bound: simular espera

In [ ]:
def tarea_io(
    identificador,
    espera=0.15
):
    time.sleep(espera)

    return {
        "id": identificador,
        "resultado": identificador ** 2
    }


inicio = time.perf_counter()

resultado = [
    tarea_io(i)
    for i in range(8)
]

tiempo_secuencial = (
    time.perf_counter()
    - inicio
)

print(
    "Tiempo secuencial:",
    round(
        tiempo_secuencial,
        3
    ),
    "s"
)

### Interpretación

La CPU pasa gran parte del tiempo esperando.

Este tipo de tarea es candidata a **threading**.

# 4. Threading sobre I/O

In [ ]:
def ejecutar_threading_io():
    with ThreadPoolExecutor(
        max_workers=4
    ) as executor:

        return list(
            executor.map(
                tarea_io,
                range(8)
            )
        )


inicio = time.perf_counter()

resultado_threads = (
    ejecutar_threading_io()
)

tiempo_threads = (
    time.perf_counter()
    - inicio
)

print(
    "Tiempo threading:",
    round(
        tiempo_threads,
        3
    ),
    "s"
)

print(
    "Speedup:",
    round(
        tiempo_secuencial
        / tiempo_threads,
        2
    ),
    "x"
)

# 5. Visualizar

In [ ]:
comparacion_io = pd.DataFrame({
    "metodo": [
        "Secuencial",
        "Threading"
    ],
    "segundos": [
        tiempo_secuencial,
        tiempo_threads
    ]
})

fig, ax = plt.subplots(figsize=(6, 4))

ax.bar(
    comparacion_io["metodo"],
    comparacion_io["segundos"]
)

ax.set(
    title="Tarea I/O-bound",
    ylabel="Segundos"
)

plt.show()

### Interpretación

El threading ayuda porque mientras un hilo espera I/O, otro puede avanzar.

No significa que threading acelere cualquier función Python.

# 6. CPU-bound: crear una tarea

In [ ]:
def tarea_cpu(
    n
):
    total = 0

    for i in range(n):
        total += (
            (i % 97)
            * (i % 89)
        )

    return total


tamanio_cpu = 2_000_000

inicio = time.perf_counter()

resultado_cpu = [
    tarea_cpu(
        tamanio_cpu
    )
    for _ in range(4)
]

tiempo_cpu_sec = (
    time.perf_counter()
    - inicio
)

print(
    "CPU secuencial:",
    round(
        tiempo_cpu_sec,
        3
    ),
    "s"
)

# 7. Probar threading en CPU-bound

In [ ]:
def ejecutar_threading_cpu():
    with ThreadPoolExecutor(
        max_workers=4
    ) as executor:

        return list(
            executor.map(
                tarea_cpu,
                [tamanio_cpu] * 4
            )
        )


inicio = time.perf_counter()

resultado_cpu_threads = (
    ejecutar_threading_cpu()
)

tiempo_cpu_threads = (
    time.perf_counter()
    - inicio
)

print(
    "CPU threading:",
    round(
        tiempo_cpu_threads,
        3
    ),
    "s"
)

### Interpretación

En CPU-bound puro de Python, threading puede no aportar mejora significativa debido al GIL.

Incluso puede empeorar por overhead.

# 8. Visualizar CPU secuencial vs threading

In [ ]:
comparacion_cpu_threads = pd.DataFrame({
    "metodo": [
        "Secuencial",
        "Threading"
    ],
    "segundos": [
        tiempo_cpu_sec,
        tiempo_cpu_threads
    ]
})

fig, ax = plt.subplots(figsize=(6, 4))

ax.bar(
    comparacion_cpu_threads["metodo"],
    comparacion_cpu_threads["segundos"]
)

ax.set(
    title="CPU-bound: threading no garantiza mejora",
    ylabel="Segundos"
)

plt.show()

# 9. Multiprocessing

Para tareas CPU-bound, los procesos pueden ejecutar código Python en intérpretes separados.

Eso evita que todos compitan por el mismo GIL.

In [ ]:
def ejecutar_multiprocessing_cpu():
    with ProcessPoolExecutor(
        max_workers=4
    ) as executor:

        return list(
            executor.map(
                tarea_cpu,
                [tamanio_cpu] * 4
            )
        )

> En notebooks, multiprocessing puede comportarse de forma diferente según sistema operativo y entorno. En Windows es especialmente importante usar código importable y `if __name__ == "__main__":` cuando se traslada a script.

# 10. Ejecutar multiprocessing de forma protegida

In [ ]:
try:
    inicio = time.perf_counter()

    resultado_cpu_mp = (
        ejecutar_multiprocessing_cpu()
    )

    tiempo_cpu_mp = (
        time.perf_counter()
        - inicio
    )

    print(
        "CPU multiprocessing:",
        round(
            tiempo_cpu_mp,
            3
        ),
        "s"
    )

except Exception as error:
    tiempo_cpu_mp = np.nan

    print(
        "Multiprocessing no disponible en este entorno:"
    )

    print(
        type(error).__name__,
        "->",
        error
    )

# 11. Comparar estrategias CPU

In [ ]:
comparacion_cpu = pd.DataFrame({
    "metodo": [
        "Secuencial",
        "Threading",
        "Multiprocessing"
    ],
    "segundos": [
        tiempo_cpu_sec,
        tiempo_cpu_threads,
        tiempo_cpu_mp
    ]
})

comparacion_cpu

## 11.1 Visualizar

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

datos_plot = (
    comparacion_cpu
    .dropna()
)

ax.bar(
    datos_plot["metodo"],
    datos_plot["segundos"]
)

ax.set(
    title="CPU-bound: comparación de estrategias",
    ylabel="Segundos"
)

plt.show()

# 12. Validar resultados

In [ ]:
print(
    "Secuencial == threading:",
    resultado_cpu
    == resultado_cpu_threads
)

if not np.isnan(
    tiempo_cpu_mp
):
    print(
        "Secuencial == multiprocessing:",
        resultado_cpu
        == resultado_cpu_mp
    )

# 13. Overhead de multiprocessing

Multiprocessing tiene costes adicionales:

- crear procesos;
- serializar argumentos;
- transferir datos;
- recoger resultados;
- coordinación.

Por eso tareas demasiado pequeñas pueden ir peor.

In [ ]:
def tarea_cpu_pequena(
    n
):
    total = 0

    for i in range(n):
        total += i * i

    return total


tareas_pequenas = [
    20_000
] * 100

# 14. Secuencial para tareas pequeñas

In [ ]:
inicio = time.perf_counter()

res_peq_sec = [
    tarea_cpu_pequena(n)
    for n in tareas_pequenas
]

t_peq_sec = (
    time.perf_counter()
    - inicio
)

print(
    round(
        t_peq_sec,
        4
    ),
    "s"
)

# 15. Multiprocessing para tareas pequeñas

In [ ]:
try:
    inicio = time.perf_counter()

    with ProcessPoolExecutor(
        max_workers=4
    ) as executor:

        res_peq_mp = list(
            executor.map(
                tarea_cpu_pequena,
                tareas_pequenas
            )
        )

    t_peq_mp = (
        time.perf_counter()
        - inicio
    )

    print(
        round(
            t_peq_mp,
            4
        ),
        "s"
    )

except Exception as error:
    t_peq_mp = np.nan
    print(
        "No disponible:",
        error
    )

### Interpretación

Si cada tarea es muy pequeña, el coste de crear y coordinar procesos puede superar el beneficio de paralelizar.

# 16. Granularidad de tareas

In [ ]:
if not np.isnan(
    t_peq_mp
):
    print(
        "Speedup:",
        round(
            t_peq_sec
            / t_peq_mp,
            2
        )
    )

Una decisión importante en paralelización es la **granularidad**:

- muchas tareas diminutas → mucho overhead;
- pocas tareas demasiado grandes → menor reparto de carga;
- tamaño intermedio → mejor aprovechamiento.

# 17. CPU-bound con NumPy

In [ ]:
vector = rng.normal(
    size=5_000_000
)

def suma_cuadrados_python(
    valores
):
    total = 0.0

    for valor in valores:
        total += (
            valor
            * valor
        )

    return total


def suma_cuadrados_numpy(
    valores
):
    return np.sum(
        valores * valores
    )

# 18. Benchmark Python vs NumPy

In [ ]:
bench_python = benchmark(
    lambda:
        suma_cuadrados_python(
            vector
        ),
    repeticiones=3,
    calentamiento=0
)

bench_numpy = benchmark(
    lambda:
        suma_cuadrados_numpy(
            vector
        ),
    repeticiones=7
)

pd.DataFrame({
    "metodo": [
        "Python",
        "NumPy"
    ],
    "media_ms": [
        bench_python["media_ms"],
        bench_numpy["media_ms"]
    ]
})

# 19. Introducción a Numba

In [ ]:
if NUMBA_DISPONIBLE:

    @njit
    def suma_cuadrados_numba(
        valores
    ):
        total = 0.0

        for valor in valores:
            total += (
                valor
                * valor
            )

        return total

    print(
        "Función compilable definida."
    )
else:
    print(
        "Numba no disponible."
    )

# 20. Primera llamada: coste de compilación

In [ ]:
if NUMBA_DISPONIBLE:
    inicio = time.perf_counter()

    resultado_primera = (
        suma_cuadrados_numba(
            vector
        )
    )

    t_primera = (
        time.perf_counter()
        - inicio
    )

    print(
        "Primera llamada:",
        round(
            t_primera * 1000,
            3
        ),
        "ms"
    )

### Interpretación

La primera llamada incluye compilación JIT.

No debe compararse directamente con una función ya “caliente”.

# 21. Segunda llamada

In [ ]:
if NUMBA_DISPONIBLE:
    inicio = time.perf_counter()

    resultado_segunda = (
        suma_cuadrados_numba(
            vector
        )
    )

    t_segunda = (
        time.perf_counter()
        - inicio
    )

    print(
        "Segunda llamada:",
        round(
            t_segunda * 1000,
            3
        ),
        "ms"
    )

# 22. Benchmark justo de Numba

In [ ]:
if NUMBA_DISPONIBLE:
    # Warm-up
    _ = suma_cuadrados_numba(
        vector[:100]
    )

    bench_numba = benchmark(
        lambda:
            suma_cuadrados_numba(
                vector
            ),
        repeticiones=7,
        calentamiento=1
    )

    comparacion_numba = pd.DataFrame({
        "metodo": [
            "Python",
            "NumPy",
            "Numba"
        ],
        "media_ms": [
            bench_python["media_ms"],
            bench_numpy["media_ms"],
            bench_numba["media_ms"]
        ]
    })

    comparacion_numba

## 22.1 Visualizar

In [ ]:
if NUMBA_DISPONIBLE:
    fig, ax = plt.subplots(figsize=(7, 4))

    ax.bar(
        comparacion_numba["metodo"],
        comparacion_numba["media_ms"]
    )

    ax.set(
        title="Python vs NumPy vs Numba",
        ylabel="Tiempo medio (ms)"
    )

    plt.show()

# 23. Validar equivalencia

In [ ]:
if NUMBA_DISPONIBLE:
    print(
        "Python vs NumPy:",
        np.isclose(
            suma_cuadrados_python(
                vector
            ),
            suma_cuadrados_numpy(
                vector
            )
        )
    )

    print(
        "NumPy vs Numba:",
        np.isclose(
            suma_cuadrados_numpy(
                vector
            ),
            suma_cuadrados_numba(
                vector
            )
        )
    )

# 24. Caso donde Numba es especialmente útil

Numba puede ser muy útil cuando:

- existe un loop numérico difícil de vectorizar;
- trabajamos con arrays NumPy;
- la función se ejecuta muchas veces;
- el coste de compilación se amortiza.

In [ ]:
def distancia_acumulada_python(
    x,
    y
):
    total = 0.0

    for i in range(
        len(x)
    ):
        dx = (
            x[i]
            - y[i]
        )

        total += (
            dx * dx
        )

    return total


if NUMBA_DISPONIBLE:

    distancia_acumulada_numba = njit(
        distancia_acumulada_python
    )

# 25. Benchmark

In [ ]:
x = rng.normal(
    size=3_000_000
)

y = rng.normal(
    size=3_000_000
)

bench_dist_py = benchmark(
    lambda:
        distancia_acumulada_python(
            x,
            y
        ),
    repeticiones=3,
    calentamiento=0
)

if NUMBA_DISPONIBLE:
    _ = distancia_acumulada_numba(
        x[:100],
        y[:100]
    )

    bench_dist_numba = benchmark(
        lambda:
            distancia_acumulada_numba(
                x,
                y
            ),
        repeticiones=7
    )

    print(
        "Python:",
        bench_dist_py["media_ms"]
    )

    print(
        "Numba:",
        bench_dist_numba["media_ms"]
    )

# 26. Numba y operaciones no soportadas

Numba no soporta de la misma manera toda la API dinámica de Python.

Por ejemplo, estructuras muy dinámicas, objetos arbitrarios o algunas operaciones de Pandas no son buenos candidatos.

In [ ]:
if NUMBA_DISPONIBLE:

    def funcion_no_ideal(
        datos
    ):
        resultado = []

        for valor in datos:
            resultado.append(
                {
                    "valor":
                        valor,
                    "texto":
                        str(valor)
                }
            )

        return resultado

    print(
        "Esta función no es un buen candidato para @njit."
    )

### Regla práctica

Numba funciona especialmente bien con:

- tipos numéricos;
- arrays NumPy;
- loops;
- operaciones matemáticas predecibles.

# 27. Paralelización con Numba

In [ ]:
if NUMBA_DISPONIBLE:

    @njit(
        parallel=True
    )
    def transformar_parallel(
        valores
    ):
        salida = np.empty_like(
            valores
        )

        for i in prange(
            len(valores)
        ):
            salida[i] = (
                valores[i] ** 2
                + np.sin(
                    valores[i]
                )
            )

        return salida

    @njit
    def transformar_numba(
        valores
    ):
        salida = np.empty_like(
            valores
        )

        for i in range(
            len(valores)
        ):
            salida[i] = (
                valores[i] ** 2
                + np.sin(
                    valores[i]
                )
            )

        return salida

# 28. Warm-up

In [ ]:
if NUMBA_DISPONIBLE:
    muestra_warm = vector[:1000]

    _ = transformar_numba(
        muestra_warm
    )

    _ = transformar_parallel(
        muestra_warm
    )

    print(
        "Warm-up completado."
    )

# 29. Benchmark Numba secuencial vs parallel

In [ ]:
if NUMBA_DISPONIBLE:

    bench_numba_seq = benchmark(
        lambda:
            transformar_numba(
                vector
            ),
        repeticiones=5
    )

    bench_numba_par = benchmark(
        lambda:
            transformar_parallel(
                vector
            ),
        repeticiones=5
    )

    comparacion_parallel = pd.DataFrame({
        "metodo": [
            "Numba secuencial",
            "Numba parallel"
        ],
        "media_ms": [
            bench_numba_seq["media_ms"],
            bench_numba_par["media_ms"]
        ]
    })

    comparacion_parallel

## 29.1 Visualizar

In [ ]:
if NUMBA_DISPONIBLE:
    fig, ax = plt.subplots(figsize=(7, 4))

    ax.bar(
        comparacion_parallel["metodo"],
        comparacion_parallel["media_ms"]
    )

    ax.set(
        title="Numba secuencial vs paralelo",
        ylabel="Tiempo medio (ms)"
    )

    plt.show()

### Interpretación

`parallel=True` tampoco garantiza mejora.

Depende de:

- tamaño;
- operación;
- número de cores;
- ancho de memoria;
- overhead de paralelización.

# 30. Validar paralelo

In [ ]:
if NUMBA_DISPONIBLE:
    seq = transformar_numba(
        vector
    )

    par = transformar_parallel(
        vector
    )

    print(
        np.allclose(
            seq,
            par
        )
    )

# 31. Escalabilidad por tamaño

In [ ]:
if NUMBA_DISPONIBLE:
    tamanos = [
        10_000,
        100_000,
        500_000,
        1_000_000,
        5_000_000
    ]

    filas = []

    for n in tamanos:
        datos_n = vector[:n]

        t_seq = benchmark(
            lambda d=datos_n:
                transformar_numba(d),
            repeticiones=5
        )["media_ms"]

        t_par = benchmark(
            lambda d=datos_n:
                transformar_parallel(d),
            repeticiones=5
        )["media_ms"]

        filas.append({
            "n": n,
            "Numba secuencial": t_seq,
            "Numba paralelo": t_par
        })

    escala_numba = pd.DataFrame(
        filas
    )

    escala_numba

# 32. Visualizar crossover

In [ ]:
if NUMBA_DISPONIBLE:
    fig, ax = plt.subplots(figsize=(8, 4))

    ax.plot(
        escala_numba["n"],
        escala_numba["Numba secuencial"],
        marker="o",
        label="Secuencial"
    )

    ax.plot(
        escala_numba["n"],
        escala_numba["Numba paralelo"],
        marker="o",
        label="Paralelo"
    )

    ax.set(
        title="Cuándo empieza a compensar paralelizar",
        xlabel="Tamaño n",
        ylabel="Tiempo medio (ms)"
    )

    ax.legend()
    plt.show()

### Interpretación

El gráfico puede revelar un punto de cruce:

- para tamaños pequeños, el overhead domina;
- para tamaños grandes, el paralelismo puede compensar.

# 33. Threading con llamadas simuladas a servicios

In [ ]:
def consultar_servicio(
    identificador
):
    time.sleep(
        0.1
    )

    return (
        identificador,
        identificador * 10
    )


ids = list(
    range(20)
)

# 34. Secuencial

In [ ]:
inicio = time.perf_counter()

res_serv_sec = [
    consultar_servicio(i)
    for i in ids
]

t_serv_sec = (
    time.perf_counter()
    - inicio
)

# 35. Thread pool

In [ ]:
inicio = time.perf_counter()

with ThreadPoolExecutor(
    max_workers=8
) as executor:

    res_serv_thread = list(
        executor.map(
            consultar_servicio,
            ids
        )
    )

t_serv_thread = (
    time.perf_counter()
    - inicio
)

print(
    "Secuencial:",
    round(
        t_serv_sec,
        3
    )
)

print(
    "Threading:",
    round(
        t_serv_thread,
        3
    )
)

# 36. Validar orden de resultados

In [ ]:
print(
    res_serv_sec
    == res_serv_thread
)

`executor.map()` conserva el orden de entrada en los resultados.

Esto puede ser importante en pipelines reproducibles.

# 37. Número de workers

In [ ]:
workers = [
    1,
    2,
    4,
    8,
    16
]

resultados_workers = []

for n_workers in workers:

    inicio = (
        time.perf_counter()
    )

    with ThreadPoolExecutor(
        max_workers=n_workers
    ) as executor:

        _ = list(
            executor.map(
                consultar_servicio,
                ids
            )
        )

    tiempo = (
        time.perf_counter()
        - inicio
    )

    resultados_workers.append({
        "workers":
            n_workers,
        "segundos":
            tiempo
    })

df_workers = pd.DataFrame(
    resultados_workers
)

df_workers

# 38. Visualizar workers

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

ax.plot(
    df_workers["workers"],
    df_workers["segundos"],
    marker="o"
)

ax.set(
    title="Número de threads vs tiempo",
    xlabel="Workers",
    ylabel="Segundos"
)

plt.show()

### Interpretación

Más workers no significa mejor rendimiento indefinidamente.

Puede aparecer saturación, límites externos o overhead.

# 39. Patrón incorrecto: paralelizar una tarea trivial

In [ ]:
def trivial(
    x
):
    return x + 1


valores_triviales = list(
    range(10_000)
)

## 39.1 Secuencial

In [ ]:
inicio = time.perf_counter()

res_trivial_sec = [
    trivial(x)
    for x in valores_triviales
]

t_trivial_sec = (
    time.perf_counter()
    - inicio
)

print(
    t_trivial_sec * 1000,
    "ms"
)

## 39.2 Threading

In [ ]:
inicio = time.perf_counter()

with ThreadPoolExecutor(
    max_workers=8
) as executor:

    res_trivial_thread = list(
        executor.map(
            trivial,
            valores_triviales
        )
    )

t_trivial_thread = (
    time.perf_counter()
    - inicio
)

print(
    t_trivial_thread * 1000,
    "ms"
)

### Interpretación

Paralelizar una operación extremadamente barata puede ser mucho peor que ejecutarla directamente.

La paralelización tiene overhead.

# 40. Matriz de decisión

In [ ]:
matriz_decision = pd.DataFrame({
    "tipo_problema": [
        "I/O-bound",
        "CPU-bound Python",
        "Numérico vectorizable",
        "Loop numérico complejo",
        "Loop numérico grande paralelizable"
    ],
    "primera_opcion": [
        "Threading",
        "Multiprocessing",
        "NumPy",
        "Numba",
        "Numba parallel / multiprocessing"
    ]
})

matriz_decision

### Importante

La tabla es una heurística inicial, no una ley.

La decisión final requiere benchmark.

# 41. Mini-práctica integrada

Vamos a imaginar un pipeline con tres fases:

1. cargar o esperar datos de varias fuentes;
2. transformar una operación numérica pesada;
3. agregar resultados.

Propuesta:

- threading para I/O;
- NumPy/Numba para transformación;
- multiprocessing solo si la etapa CPU no es vectorizable y el volumen lo justifica.

# 42. Checklist de concurrencia

- [ ] ¿La tarea es I/O-bound o CPU-bound?
- [ ] ¿Tengo un baseline?
- [ ] ¿La operación puede vectorizarse?
- [ ] ¿NumPy ya resuelve el problema?
- [ ] ¿Numba puede compilar la función?
- [ ] ¿He separado warm-up de ejecución?
- [ ] ¿El problema es suficientemente grande?
- [ ] ¿He medido overhead?
- [ ] ¿Los procesos requieren serializar datos grandes?
- [ ] ¿El número de workers está justificado?
- [ ] ¿He probado varios tamaños?
- [ ] ¿He validado resultados?
- [ ] ¿La mejora compensa la complejidad?

# 43. Ejercicios finales

## Ejercicio 1 — I/O-bound
Simula 20 tareas con `sleep()` y compara secuencial vs threading.

## Ejercicio 2 — Workers
Prueba 1, 2, 4, 8 y 16 threads y representa el tiempo.

## Ejercicio 3 — CPU-bound
Crea una función con un loop intensivo y compara secuencial vs threading.

## Ejercicio 4 — Multiprocessing
Repite el ejercicio anterior con procesos.

## Ejercicio 5 — Overhead
Demuestra un caso donde multiprocessing sea peor por tener tareas demasiado pequeñas.

## Ejercicio 6 — Validación
Comprueba que secuencial, threads y procesos devuelven los mismos resultados.

## Ejercicio 7 — NumPy
Vectoriza un loop numérico.

## Ejercicio 8 — Numba
Compila el loop original con `@njit`.

## Ejercicio 9 — Warm-up
Compara primera llamada y llamadas posteriores.

## Ejercicio 10 — Numba vs NumPy
Compara ambas técnicas con una operación numérica.

## Ejercicio 11 — `prange`
Implementa una transformación paralela con Numba.

## Ejercicio 12 — Crossover
Busca el tamaño a partir del cual `parallel=True` empieza a compensar.

## Ejercicio 13 — Incompatibilidad
Construye una función Python que no sea buena candidata para Numba y explica por qué.

## Ejercicio 14 — Tarea trivial
Demuestra que paralelizar una operación muy barata empeora rendimiento.

## Ejercicio 15 — Matriz de decisión
Clasifica diez tareas como I/O-bound, CPU-bound, vectorizables o compilables.

## Ejercicio 16 — Reto final
Construye un pipeline con:

1. fase I/O simulada;
2. transformación CPU-bound;
3. baseline secuencial;
4. threading;
5. multiprocessing;
6. NumPy;
7. Numba;
8. Numba paralelo;
9. benchmarks;
10. varios tamaños;
11. varios workers;
12. validación;
13. speedups;
14. interpretación;
15. recomendación final.

# 44. Preguntas de reflexión

1. ¿Qué diferencia existe entre I/O-bound y CPU-bound?
2. ¿Por qué threading ayuda en I/O?
3. ¿Por qué threading no siempre acelera CPU-bound Python?
4. ¿Qué coste tiene multiprocessing?
5. ¿Qué significa granularidad de tarea?
6. ¿Por qué demasiados procesos pueden empeorar rendimiento?
7. ¿Qué es JIT?
8. ¿Por qué la primera llamada de Numba es más lenta?
9. ¿Cuándo puede Numba superar a Python puro?
10. ¿Numba sustituye a NumPy?
11. ¿Qué hace `parallel=True`?
12. ¿Qué papel tiene `prange`?
13. ¿Por qué paralelizar una tarea trivial es mala idea?
14. ¿Qué significa crossover de rendimiento?
15. ¿Cuál debería ser el orden correcto: paralelizar o medir?